In [ ]:
%pip install langchain-text-splitters

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np

from snowflake.snowpark.context import get_active_session
from langchain_text_splitters import RecursiveCharacterTextSplitter

import utils

In [ ]:
%%sql -r directory_results
-- SELECT * FROM DIRECTORY(@PROJECT_DB.PUBLIC.FILES);

In [ ]:
%%sql -r dataframe_5
/*
SELECT 
    relative_path AS file_name,
    CASE
        WHEN FL_IS_VIDEO(TO_FILE('@PROJECT_DB.PUBLIC.FILES', relative_path))
          OR FL_IS_AUDIO(TO_FILE('@PROJECT_DB.PUBLIC.FILES', relative_path))
            THEN AI_TRANSCRIBE(TO_FILE('@PROJECT_DB.PUBLIC.FILES', relative_path)):"text"::TEXT
        WHEN relative_path ILIKE '%.xlsx' OR relative_path ILIKE '%.xls'
            THEN AI_EXTRACT(
                    file => TO_FILE('@PROJECT_DB.PUBLIC.FILES', relative_path),
                     responseFormat => {'content': 'Extract all the text content from this spreadsheet'}
                     ):"response":"content"::TEXT
        ELSE AI_PARSE_DOCUMENT(TO_FILE('@PROJECT_DB.PUBLIC.FILES', relative_path)):"content"::TEXT
    END AS full_content
FROM DIRECTORY(@PROJECT_DB.PUBLIC.FILES)
*/

In [ ]:
session = get_active_session()

In [ ]:
# 1. Configurer le découpeur de texte
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len
)

In [ ]:
# 2. Récupérer les documents bruts extraits par Snowflake
raw_docs = session.sql("""
    SELECT 
        relative_path AS file_name,
        CASE
            WHEN FL_IS_VIDEO(
                TO_FILE('@PROJECT_DB.PUBLIC.FILES', relative_path)
            )
            OR FL_IS_AUDIO(
                TO_FILE('@PROJECT_DB.PUBLIC.FILES', relative_path)
            )
                THEN AI_TRANSCRIBE(
                    TO_FILE('@PROJECT_DB.PUBLIC.FILES', relative_path)
                ):"text"::TEXT

            WHEN relative_path ILIKE '%.xlsx'
              OR relative_path ILIKE '%.xls'
                THEN AI_PARSE_DOCUMENT(
                    TO_FILE('@PROJECT_DB.PUBLIC.FILES', relative_path)
                ):"content"::TEXT

            ELSE AI_PARSE_DOCUMENT(
                TO_FILE('@PROJECT_DB.PUBLIC.FILES', relative_path)
            ):"content"::TEXT
        END AS full_content

    FROM DIRECTORY(@PROJECT_DB.PUBLIC.FILES)

    WHERE LOWER(relative_path) NOT LIKE '%extraction baf09.xlsx'
""").collect()

In [ ]:
%%sql -r dataframe_1
-- 3. Créer la table sql
CREATE OR REPLACE TABLE PROJECT_DB.PUBLIC.BASE_CONNAISSANCE_RAG (
    content TEXT, 
    file_name TEXT,
    source_type TEXT,
    embeddings VECTOR(FLOAT, 768) DEFAULT AI_EMBED('snowflake-arctic-embed-m-v1.5', content)
);

In [ ]:
# 4. Appliquer le chunking et insérer chaque morceau séparément
for row in raw_docs:
    file_name = row["FILE_NAME"]
    content = row["FULL_CONTENT"]
    
    if content:
        # Découpage du texte long en liste de chunks
        chunks = text_splitter.split_text(content)
        
        for idx, chunk in enumerate(chunks):
            # Insertion de chaque chunk avec un identifiant de morceau
            chunk_file_name = f"{file_name}#chunk{idx+1}"
            
            # Injection sécurisée du texte
            session.sql("""
                INSERT INTO PROJECT_DB.PUBLIC.BASE_CONNAISSANCE_RAG (CONTENT, FILE_NAME, SOURCE_TYPE)
                VALUES (?, ?, 'document')
            """, params=[chunk, chunk_file_name]).collect()

print("Chunking terminé avec succès !")

In [ ]:
# 5. Ajouter les fiches FAQ de KB_QA dans la base documentaire

session.sql("""
    INSERT INTO PROJECT_DB.PUBLIC.BASE_CONNAISSANCE_RAG
        (CONTENT, FILE_NAME, SOURCE_TYPE)

    SELECT
        'Question: ' || QUESTION ||
        '\nAnswer: ' || ANSWER AS CONTENT,

        ID AS FILE_NAME,

        'faq' AS SOURCE_TYPE

    FROM PROJECT_DB.PUBLIC.KB_QA
""").collect()

print("KB_QA ajoutée à la base documentaire.")

In [ ]:
%%sql -r dataframe_4
select * from PROJECT_DB.PUBLIC.BASE_CONNAISSANCE_RAG;

In [ ]:
%%sql -r validation_results
--Validation
SELECT
    SOURCE_TYPE,
    COUNT(*) AS NB_PASSAGES
FROM PROJECT_DB.PUBLIC.BASE_CONNAISSANCE_RAG
GROUP BY SOURCE_TYPE
ORDER BY SOURCE_TYPE;

In [ ]:
%%sql -r dataframe_2
SELECT *
FROM PROJECT_DB.PUBLIC.BASE_CONNAISSANCE_RAG
WHERE LOWER(FILE_NAME) LIKE '%extraction baf09.xlsx%';

In [ ]:
%%sql -r dataframe_3
